# 技能推荐：交互式实验

本 notebook 把中文镜像站中的 [Cookbook](/cookbooks/skill_suggestion/) 改写成可以逐格运行、修改输入并观察结果的最小实验。
先用 Noul 宽召回，再用 Choice 选择主技能。

运行方式与 `../03_架构模式/01_架构模式.ipynb` 一致：有有效的 `TYPESAFE_API_KEY` 时调用真实的
TypeSafe API；没有 Key 或返回 401 时使用内置的离线示例答案。后续代码不区分两种模式，便于先学习
控制流，再切换到真实模型观察概率和置信度。

> 学习提示：先顺序运行全部单元格，再回到“定义 state”或“定义问题”的单元格修改内容，重新运行后面的单元格。
> API Key 只从环境变量读取，不能写进 notebook。


## 0. 准备

### 0.1 安装依赖

In [1]:
%pip install -q -U typesafe-sdk

Note: you may need to restart the kernel to use updated packages.


### 0.2 创建客户端

In [2]:
import os
import statistics
import time
from pprint import pprint

from typesafe_sdk import (
    Choice,
    Score,
    Noul,
    TypeSafeClient,
    TypeSafeAuthenticationError,
)

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
client = TypeSafeClient(api_key=API_KEY, model="jev-latest") if API_KEY else None
print("客户端已创建：模型=jev-latest，Key=", "已配置" if API_KEY else "未配置（将使用离线示例）")


客户端已创建：模型=jev-latest，Key= 已配置


### 0.3 离线响应与统一调用入口

In [3]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for key, value in values.items():
            setattr(self, key, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.model = "jev-latest（离线示例）"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)


class TS:
    offline = False
    _warned = False

    @classmethod
    def call(cls, state, questions, offline_answers):
        if client is None:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)
        try:
            return client.system_one(state, questions)
        except TypeSafeAuthenticationError:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)


def answer_line(name, answer):
    if answer.type == "noul":
        return f"{name}: noul={answer.noul:.2f}"
    if answer.type == "choice":
        return f"{name}: choice={answer.choice} confidence={answer.confidence:.2f}"
    return f"{name}: score={answer.score:.2f} confidence={answer.confidence:.2f}"


print("模式：", "离线示例" if TS.offline else "真实 API（首次调用后确定）")


模式： 真实 API（首次调用后确定）


### 0.4 连通性测试

In [4]:
if client is None:
    TS.offline = True
    print("⚠️ API Key 未设置，后续单元格使用离线示例。")
else:
    try:
        ping = client.system_one("你好", {"is_greeting": Noul(instructions="这段文字是在打招呼吗？")})
        print("✅ API 连通正常，后续单元格会使用真实结果。")
    except TypeSafeAuthenticationError:
        TS.offline = True
        print("⚠️ API Key 无效，后续单元格使用离线示例。")


✅ API 连通正常，后续单元格会使用真实结果。


## 1. 技能推荐

先对技能名册中的每一项做廉价的相关性判断，再从候选中选出最合适的技能。推荐结果仍由代码根据
置信度决定是否展示。


### 1.1 定义用户请求和技能名册

In [5]:
REQUEST = "帮我把本周会议纪要整理成行动项，并安排下周跟进会议。"
SKILLS = {
    "calendar": "创建和修改日历日程，查询忙闲和会议室",
    "meeting_summary": "整理会议纪要，提取决定、行动项和负责人",
    "task": "创建待办任务，分配成员并跟踪状态",
    "mail": "搜索、起草、回复和发送邮件",
}
print("技能名册已定义：技能数=", len(SKILLS))


技能名册已定义：技能数= 4


### 1.2 先做宽召回：每个技能一个 Noul

In [6]:
OFFLINE = {"calendar": .88, "meeting_summary": .94, "task": .79, "mail": .18}
ranked = []
for name, description in SKILLS.items():
    response = TS.call({"request": REQUEST, "skill": description},
                       {"relevant": Noul(instructions="这个技能是否可能帮助完成用户请求？")},
                       {"relevant": _FakeAnswer("noul", noul=OFFLINE[name])})
    probability = response.nouls["relevant"].noul
    ranked.append((probability, name))
    print(f"{probability:.2f}  {name}: {description}")
ranked.sort(reverse=True)
SHORTLIST = [name for _, name in ranked[:3]]
print("候选前三名：", SHORTLIST)


0.89  calendar: 创建和修改日历日程，查询忙闲和会议室


0.98  meeting_summary: 整理会议纪要，提取决定、行动项和负责人


0.96  task: 创建待办任务，分配成员并跟踪状态


0.88  mail: 搜索、起草、回复和发送邮件
候选前三名： ['meeting_summary', 'task', 'calendar']


### 1.3 在候选中选择主技能

In [7]:
criteria = {name: SKILLS[name] for name in SHORTLIST}
question = {"best_skill": Choice(instructions="哪个候选技能最适合先处理请求？", criteria=criteria)}
offline_choice = SHORTLIST[1] if len(SHORTLIST) > 1 else SHORTLIST[0]
response = TS.call({"request": REQUEST, "shortlist": criteria}, question,
                   {"best_skill": _FakeAnswer("choice", choice=offline_choice, confidence=.86,
                                              probabilities={offline_choice: .86})})
answer = response.choices["best_skill"]
print(f"推荐技能：{answer.choice}（confidence={answer.confidence:.2f}）")
if answer.confidence < 0.60:
    print("→ 置信度不足，交给人工选择。")
else:
    print("→ 先加载技能：", SKILLS[answer.choice])


推荐技能：meeting_summary（confidence=1.00）
→ 先加载技能： 整理会议纪要，提取决定、行动项和负责人


观察：宽召回使用 Noul 判断“是否值得考虑”，候选重排使用 Choice 做相对选择；两者回答的是不同问题。

## 知识补充
- **两段式 = 扇出的缩影**：Noul 宽召回（问一圈"相关吗"）+ Choice 精选，正是官方 Patterns 里 speculative fan-out 的最小形态。
- **为什么先 Noul 后 Choice**：Choice 的选项数决定 token 与区分难度；先用便宜的 noul 把候选缩到 2-3 个，Choice 又快又准。
- **进阶阅读**：完整扇出与置信度门控见 `../03_架构模式/01_架构模式.ipynb`。

## 小结

这本 notebook 的边界很清楚：TypeSafe 只负责受限、可编程的判断；排序、阈值、分组、重建文本和
函数分派都由 Python 代码完成。修改输入或问题后重新运行，就能观察“模型答案 → 确定性代码”的变化。
